# Clase 017 — NumPy: broadcasting

**Parte 0** · VanderPlas cap. 2 § 2.5.

> 🎯 El mecanismo por el que NumPy opera arrays de shapes distintos sin copiar datos.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1️⃣ Las 3 reglas de broadcasting

**Regla 1**: si los arrays tienen distinta cantidad de dimensiones, se **rellena con 1s a la izquierda** la shape del menor.

**Regla 2**: si en alguna dim los tamaños difieren y **uno es 1**, ese se **estira** (virtualmente) al otro.

**Regla 3**: si en alguna dim los tamaños difieren y **ninguno es 1**, **error**.

Ejemplos:
```
(3,4) + (4,)        → (3,4) + (1,4)  → (3,4) + (3,4)  ✅
(3,4) + (3,1)        → ya alineadas   → (3,4) + (3,4)  ✅
(3,4) + (3,)        → (3,4) + (1,3)  → (3,4) + (3,3)  ❌ regla 3
```

## 2️⃣ Vector + matriz: el caso más común

In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6]])     # (2, 3)
v = np.array([10, 20, 30])               # (3,)
print('M + v (vector como fila):')
print(M + v)
print()

w = np.array([[100], [200]])             # (2, 1)
print('M + w (vector como columna):')
print(M + w)

## 3️⃣ `np.newaxis` (alias `None`) — promover un vector

A veces necesitas decir explícitamente "este vector es una fila" o "es una columna":

In [ ]:
v = np.array([1, 2, 3])    # (3,)

# Como fila (1, 3)
fila = v[np.newaxis, :]
print(f'fila shape: {fila.shape}')   # (1, 3)

# Como columna (3, 1) — sintaxis equivalente con None
col = v[:, None]
print(f'col shape : {col.shape}')    # (3, 1)

# Outer product: (3,1) * (1,4) → (3,4)
a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])
outer = a[:, None] * b[None, :]
print(f'\nouter shape: {outer.shape}')
print(outer)

## 4️⃣ Caso canónico — estandarizar features

Matriz `X` con shape `(n_muestras, n_features)`. Queremos centrar y escalar por feature:

```python
μ = X.mean(axis=0)     # (n_features,)
σ = X.std(axis=0)      # (n_features,)
Z = (X - μ) / σ        # broadcasting: (n,d) - (d,) → (n,d)
```

In [ ]:
X = rng.normal(loc=[5, 10, 100], scale=[1, 2, 50], size=(100, 3))
print(f'X.shape  : {X.shape}')
print(f'media    : {X.mean(axis=0).round(2)}')
print(f'std      : {X.std(axis=0).round(2)}')

# Estandariza en una línea
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print(f'\nZ.mean   : {Z.mean(axis=0).round(4)}  (≈ 0)')
print(f'Z.std    : {Z.std(axis=0).round(4)}  (≈ 1)')

## 5️⃣ Distance matrix — sin loop

Dados `n` puntos en `d` dimensiones, queremos matriz `n×n` con distancias euclídeas. Usando broadcasting:

```python
diff = X[:, None, :] - X[None, :, :]    # (n, 1, d) - (1, n, d) → (n, n, d)
dist = np.sqrt((diff ** 2).sum(axis=2)) # (n, n)
```

In [ ]:
puntos = rng.normal(0, 1, (5, 2))   # 5 puntos 2D
diff = puntos[:, None, :] - puntos[None, :, :]
print(f'diff.shape: {diff.shape}')   # (5, 5, 2)
dist = np.sqrt((diff ** 2).sum(axis=2))
print(f'\ndist matrix (5x5):')
print(dist.round(2))
print(f'\ndiagonal (cada punto consigo): {np.diag(dist)}  (≈ 0)')

## 6️⃣ Cuando broadcasting falla

El error típico:

```
ValueError: operands could not be broadcast together with shapes (3,4) (4,3)
```

**Cómo leerlo**:
1. Alinea las shapes a la derecha.
2. En cada columna alineada, debe haber `==` o uno `1`.
3. Si no, falla.

Para `(3,4)` y `(4,3)`:
```
(3, 4)
(4, 3)
```
Columna derecha: 4 vs 3 → distintos y ninguno es 1 → falla.

In [ ]:
try:
    np.ones((3, 4)) + np.ones((4, 3))
except ValueError as e:
    print(f'ValueError: {e}')
    print('Lo correcto: transponer uno, o asegurar dims compatibles.')
    print(f'Con transpose: {(np.ones((3,4)) + np.ones((4,3)).T).shape}')

## ✅ Checklist

- [ ] Sé las 3 reglas de broadcasting de memoria
- [ ] Predigo la shape del resultado antes de ejecutar
- [ ] Estandarizo una matriz por columna en una línea
- [ ] Uso `[:, None]` para promover a columna
- [ ] Leo y diagnostico ValueError de broadcasting

## 📝 Homework

Ver `README.md`. Predicción de shapes, estandarización, distance matrix sin loop, diagnóstico de error.

## 📖 Definiciones y características

**Broadcasting**

Mecanismo por el que NumPy opera arrays de shapes distintos **sin copiar memoria**, estirando virtualmente las dimensiones de tamaño 1. Lo que hace posible `X - X.mean(axis=0)` (centrado por columna) en una línea sin loops.

**Regla 1 — padding por la izquierda**

Si los arrays tienen distinta cantidad de dimensiones, la shape del menor se rellena con `1`s a la izquierda. `(4,)` operado con `(3, 4)` se trata como `(1, 4)` vs `(3, 4)`.

**Regla 2 — estirar dim 1**

En cada dimensión donde los tamaños difieren, si uno es `1` se estira al otro. `(3, 1)` y `(3, 4)` → ambos `(3, 4)` (la primera se estira en eje 1).

**Regla 3 — fallo**

Si en alguna dimensión los tamaños son distintos y **ninguno es 1**, lanza `ValueError: operands could not be broadcast together`. No hay forma de inferir qué hacer.

**`np.newaxis` (alias `None`)**

Inserta una dimensión de tamaño 1 donde lo pongas. `v[:, None]` convierte vector `(3,)` en columna `(3, 1)`. Crítico para forzar broadcasting en la dirección correcta.

**Outer product vía broadcasting**

`a[:, None] * b[None, :]` produce matriz `(len(a), len(b))` con todos los productos par a par — equivalente a `np.outer(a, b)` pero usando broadcasting puro.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `ValueError: operands could not be broadcast together with shapes (3,4) (4,3)` | Las shapes alineadas por la derecha no son compatibles. **Fix**: lee el error literal, alinea shapes a la derecha mentalmente, transpone (`.T`) uno o ajusta con `newaxis`. |
| Resté `M.mean(axis=0)` y los promedios quedaron MAL | `mean(axis=0)` devuelve shape `(n_cols,)` — se broadcastea como FILA. Si querías restar por fila, usa `M.mean(axis=1)[:, None]` para que se broadcastee como COLUMNA. |
| `a + b` con shapes `(3,)` y `(3,)` da escalar (suma punto) | **No** — da array `(3,)` elementwise. Producto punto es `np.dot(a, b)` o `a @ b`. Confundir esto es el bug #1 al empezar. |
| Memoria explota en una operación 'inocente' | `X[:, None, :] - X[None, :, :]` produce array `(N, N, D)`. Si N=10000, son 100M × D elementos. **Fix**: usa `scipy.spatial.distance.cdist` o procesa por chunks. |
| `a[None] + b` no se broadcastea como espero | `a[None]` añade dim al inicio. Quizás querías `a[:, None]` (al medio) o `a[None, :]` (explícito). Usa `print(arr.shape)` siempre antes de operar. |

## ❓ Preguntas frecuentes

**❓ ¿Cómo predigo la shape del resultado?**

(1) Alinea las shapes por la derecha. (2) En cada columna alineada: si son iguales o uno es 1, OK; si no, error. (3) Resultado: por dimensión, toma el `max` de las dos.

**❓ ¿Broadcasting copia memoria?**

**No** — es virtual. NumPy itera con strides 0 en las dims estiradas. Por eso es tan eficiente: cero alloc extra (excepto el array resultado).

**❓ ¿`X - X.mean(0)` o `X - X.mean(0, keepdims=True)`?**

Para 2D ambos funcionan (broadcasting alinea). En 3D+, `keepdims=True` preserva la dimensión como `1` y evita confusiones. Recomendado en general.

**❓ ¿`np.newaxis` o `None`?**

Aliases — `arr[:, None]` y `arr[:, np.newaxis]` son idénticos. `None` es más conciso; muchos prefieren `np.newaxis` por explicitud.

**❓ ¿Qué hago si broadcasting no me sirve?**

Operaciones que no se ajustan a las reglas: usa `np.einsum` (más expresivo), `np.tensordot`, o reshape explícito. Como último recurso, loop Python — pero busca librería específica antes.

## 🔗 Referencias

- VanderPlas cap. 2 § 2.5
- [Broadcasting docs](https://numpy.org/doc/stable/user/basics.broadcasting.html)

➡️ **Siguiente:** [018 — Boolean masks y fancy indexing](../018-numpy-boolean-masks-y-fancy-indexing/README.md)